# Task 1: Data Exploration and Enrichment

## Forecasting Financial Inclusion in Ethiopia

This notebook explores the unified financial inclusion dataset, validates its schema, identifies data-quality issues, and enriches it with additional observations, events, and impact relationships relevant to forecasting Access and Usage.

### Objectives

1. Load and inspect all provided datasets.
2. Understand the unified record structure.
3. Validate categorical fields using the reference codes.
4. Review observations, events, targets, and impact links.
5. Identify missing or inconsistent information.
6. Add relevant records using credible sources.
7. Export a cleaned and enriched analysis-ready dataset.

In [1]:
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_DATA_PATH = RAW_DATA_DIR / "ethiopia_fi_unified_data.xlsx"
REFERENCE_CODES_PATH = RAW_DATA_DIR / "reference_codes.xlsx"
GUIDE_PATH = RAW_DATA_DIR / "Additional Data Points Guide.xlsx"

print("Project root:", PROJECT_ROOT)
print("Unified data exists:", UNIFIED_DATA_PATH.exists())
print("Reference codes exist:", REFERENCE_CODES_PATH.exists())
print("Enrichment guide exists:", GUIDE_PATH.exists())

Project root: c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast
Unified data exists: True
Reference codes exist: True
Enrichment guide exists: True


In [3]:
unified_workbook = pd.ExcelFile(UNIFIED_DATA_PATH)
reference_workbook = pd.ExcelFile(REFERENCE_CODES_PATH)
guide_workbook = pd.ExcelFile(GUIDE_PATH)

print("Unified workbook sheets:")
print(unified_workbook.sheet_names)

print("\nReference-code workbook sheets:")
print(reference_workbook.sheet_names)

print("\nAdditional Data Points Guide sheets:")
print(guide_workbook.sheet_names)

Unified workbook sheets:
['ethiopia_fi_unified_data', 'Impact_sheet']

Reference-code workbook sheets:
['reference_codes']

Additional Data Points Guide sheets:
['A. Alternative Baselines', 'B. Direct Corrln', 'C. Indirect Corrln', 'D. Market Naunces']


In [4]:
main_df = pd.read_excel(
    UNIFIED_DATA_PATH,
    sheet_name="ethiopia_fi_unified_data"
)

impact_df = pd.read_excel(
    UNIFIED_DATA_PATH,
    sheet_name="Impact_sheet"
)

reference_codes_df = pd.read_excel(
    REFERENCE_CODES_PATH,
    sheet_name="reference_codes"
)

alternative_baselines_df = pd.read_excel(
    GUIDE_PATH,
    sheet_name="A. Alternative Baselines",
    header=8
)

direct_indicators_df = pd.read_excel(
    GUIDE_PATH,
    sheet_name="B. Direct Corrln",
    header=9
)

indirect_indicators_df = pd.read_excel(
    GUIDE_PATH,
    sheet_name="C. Indirect Corrln",
    header=9
)

market_nuances_df = pd.read_excel(
    GUIDE_PATH,
    sheet_name="D. Market Naunces",
    header=7
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [5]:
datasets = {
    "Main data": main_df,
    "Impact links": impact_df,
    "Reference codes": reference_codes_df,
    "Alternative baselines": alternative_baselines_df,
    "Direct indicators": direct_indicators_df,
    "Indirect indicators": indirect_indicators_df,
    "Market nuances": market_nuances_df,
}

for name, dataframe in datasets.items():
    print(f"{name:<25}: {dataframe.shape[0]} rows × {dataframe.shape[1]} columns")

Main data                : 43 rows × 34 columns
Impact links             : 14 rows × 35 columns
Reference codes          : 71 rows × 4 columns
Alternative baselines    : 10 rows × 8 columns
Direct indicators        : 19 rows × 6 columns
Indirect indicators      : 16 rows × 6 columns
Market nuances           : 10 rows × 7 columns


In [6]:
all_columns = sorted(set(main_df.columns).union(set(impact_df.columns)))

main_aligned = main_df.reindex(columns=all_columns)
impact_aligned = impact_df.reindex(columns=all_columns)

unified_df = pd.concat(
    [main_aligned, impact_aligned],
    ignore_index=True
)

print("Combined unified dataset shape:", unified_df.shape)
print("Total records:", len(unified_df))

Combined unified dataset shape: (57, 35)
Total records: 57


In [7]:
record_type_counts = (
    unified_df["record_type"]
    .value_counts(dropna=False)
    .rename_axis("record_type")
    .reset_index(name="record_count")
)

record_type_counts

,record_type,record_count
0,observation,30
1,impact_link,14
2,event,10
3,target,3


In [8]:
print("Duplicate record IDs:", unified_df["record_id"].duplicated().sum())
print("Missing record IDs:", unified_df["record_id"].isna().sum())
print("Missing record types:", unified_df["record_type"].isna().sum())

event_records = unified_df[unified_df["record_type"] == "event"]
impact_records = unified_df[unified_df["record_type"] == "impact_link"]

print("\nEvents with incorrectly populated pillar:")
print(event_records[event_records["pillar"].notna()][
    ["record_id", "indicator", "category", "pillar"]
])

print("\nImpact links with missing parent ID:")
print(impact_records[impact_records["parent_id"].isna()][
    ["record_id", "indicator", "parent_id"]
])

Duplicate record IDs: 0
Missing record IDs: 0
Missing record types: 0

Events with incorrectly populated pillar:
Empty DataFrame
Columns: [record_id, indicator, category, pillar]
Index: []

Impact links with missing parent ID:
Empty DataFrame
Columns: [record_id, indicator, parent_id]
Index: []
